In [3]:
import pandas as pd
import numpy as np 
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoNormal
import pyro.optim as optim


## Preprocessing

In [4]:
# import dataset
df = pd.read_csv("data/Cleaned_dataset.csv")

In [5]:
# print features
print(df.columns)

Index(['Date_of_journey', 'Journey_day', 'Airline', 'Flight_code', 'Class',
       'Source', 'Departure', 'Total_stops', 'Arrival', 'Destination',
       'Duration_in_hours', 'Days_left', 'Fare'],
      dtype='object')


In [6]:
# select the features we want to use for the model
df = df[["Journey_day", "Airline", "Source", "Departure", "Total_stops",
         "Arrival", "Destination", "Duration_in_hours", "Days_left", "Fare"]]

# convert fare currency from INR to USD
df[["Fare"]] = (df[["Fare"]] * 0.01066).round(0).astype("Int64")

print(df.dtypes)

Journey_day           object
Airline               object
Source                object
Departure             object
Total_stops           object
Arrival               object
Destination           object
Duration_in_hours    float64
Days_left              int64
Fare                   Int64
dtype: object


In [7]:
# use integer encoding for categorical features
df["Journey_day"] = df["Journey_day"].astype("category").cat.codes
df["Airline"] = df["Airline"].astype("category").cat.codes
df["Source"] = df["Source"].astype("category").cat.codes
df["Departure"] = df["Departure"].astype("category").cat.codes
df["Total_stops"] = df["Total_stops"].astype("category").cat.codes
df["Arrival"] = df["Arrival"].astype("category").cat.codes
df["Destination"] = df["Destination"].astype("category").cat.codes

# list of the number of unique values for each categorical variable
cardinalities = [len(df["Journey_day"].unique()), len(df["Airline"].unique()), len(df["Source"].unique()), len(df["Departure"].unique()),
        len(df["Total_stops"].unique()), len(df["Arrival"].unique()), len(df["Destination"].unique())]


# split into training and test set
train_fraction = 0.8
df_shuffled = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

split_idx = int(len(df_shuffled) * train_fraction)

df_train = df_shuffled.iloc[:split_idx].copy()
df_test = df_shuffled.iloc[split_idx:].copy()

# compute scaling to standardize numerical features based on the training set only
duration_mean = df_train["Duration_in_hours"].mean()
duration_std = df_train["Duration_in_hours"].std()
days_left_mean = df_train["Days_left"].mean()
days_left_std = df_train["Days_left"].std()

# compute scaling for target variable (fare)
fare_mean = df_train["Fare"].mean()
fare_std = df_train["Fare"].std()

# apply scaling to training set
df_train["Duration_in_hours"] = (df_train["Duration_in_hours"] - duration_mean) / duration_std
df_train["Days_left"] = (df_train["Days_left"] - days_left_mean) / days_left_std
df_train["Fare"] = (df_train["Fare"] - fare_mean) / fare_std

# apply scaling to test set
df_test["Duration_in_hours"] = (df_test["Duration_in_hours"] - duration_mean) / duration_std
df_test["Days_left"] = (df_test["Days_left"] - days_left_mean) / days_left_std
df_test["Fare"] = (df_test["Fare"] - fare_mean) / fare_std


# create torch tensors for each categorical variable for both training and test set
journey_day_tensor_train = torch.tensor(df_train["Journey_day"].values, dtype=torch.long)
journey_day_tensor_test = torch.tensor(df_test["Journey_day"].values, dtype=torch.long)
airline_tensor_train = torch.tensor(df_train["Airline"].values, dtype=torch.long)
airline_tensor_test = torch.tensor(df_test["Airline"].values, dtype=torch.long)
source_tensor_train = torch.tensor(df_train["Source"].values, dtype=torch.long)
source_tensor_test = torch.tensor(df_test["Source"].values, dtype=torch.long)
departure_tensor_train = torch.tensor(df_train["Departure"].values, dtype=torch.long)
departure_tensor_test = torch.tensor(df_test["Departure"].values, dtype=torch.long)
total_stops_tensor_train = torch.tensor(df_train["Total_stops"].values, dtype=torch.long)
total_stops_tensor_test = torch.tensor(df_test["Total_stops"].values, dtype=torch.long)
arrival_tensor_train = torch.tensor(df_train["Arrival"].values, dtype=torch.long)
arrival_tensor_test = torch.tensor(df_test["Arrival"].values, dtype=torch.long)
destination_tensor_train = torch.tensor(df_train["Destination"].values, dtype=torch.long)
destination_tensor_test = torch.tensor(df_test["Destination"].values, dtype=torch.long)

# create torch tensors for each numerical variable for both training and test set
duration_in_hours_tensor_train = torch.tensor(df_train["Duration_in_hours"].values, dtype=torch.float)
duration_in_hours_tensor_test = torch.tensor(df_test["Duration_in_hours"].values, dtype=torch.float)
days_left_tensor_train = torch.tensor(df_train["Days_left"].values, dtype=torch.float)
days_left_tensor_test = torch.tensor(df_test["Days_left"].values, dtype=torch.float)

# create torch tensor for target variable for both training and test set
fare_tensor_train = torch.tensor(df_train["Fare"].values, dtype=torch.float)
fare_tensor_test = torch.tensor(df_test["Fare"].values, dtype=torch.float)


# lists of each type of feature (categorical and numerical)
cat_tensors_train = [journey_day_tensor_train, airline_tensor_train, source_tensor_train, departure_tensor_train, total_stops_tensor_train,
                     arrival_tensor_train, destination_tensor_train]
cat_tensors_test = [journey_day_tensor_test, airline_tensor_test, source_tensor_test, departure_tensor_test, total_stops_tensor_test,
                     arrival_tensor_test, destination_tensor_test]
num_tensors_train = [duration_in_hours_tensor_train, days_left_tensor_train]
num_tensors_test = [duration_in_hours_tensor_test, days_left_tensor_test]